# **GPT-2 with Autonomous Architecture Generation (AAG), fine-tuned on Alpaca**

Mohammad Al Dridi

Replaces the MLP in each GPT-2 transformer block with a *chunked* expert bank.
Instead of the router picking one of four whole experts, it slices the MLP's
weight matrix into `NUM_CHUNKS` horizontal bands and picks one of `NUM_OPTIONS`
versions **per band, independently**. A token's effective weight matrix is
assembled from the winning bands.

That turns 4 routing choices into `4 ** NUM_CHUNKS` per projection, while still
executing exactly one matrix's worth of arithmetic per token.

Runtime -> Change runtime type -> **L4 GPU**, and enable **background execution**.

## Import

In [1]:
import copy
import math
import os
import time
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import (AutoTokenizer, GPT2LMHeadModel, Trainer,
                          TrainingArguments)
from transformers.activations import ACT2FN
from datasets import load_dataset
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

device: cuda
gpu: NVIDIA L4


## Config

In [2]:
# Architecture
NUM_CHUNKS  = 8       # bands the MLP weight matrix is sliced into
NUM_OPTIONS = 4       # candidate versions stored per band
AUX_COEF    = 0.05    # load-balancing loss weight

# How the chosen option per band is executed. The two are numerically
# identical -- verified to float32 epsilon on the forward pass, on every
# gradient, and on which options receive gradient at all. They differ only
# in speed, and which wins is hardware-dependent, so the cell below
# measures both on this GPU before you commit hours to a run.
#   'fused'  : one big matmul over all options, then gather the winner.
#              num_options times the arithmetic, but one kernel, no syncs.
#   'masked' : only the needed arithmetic, but num_chunks * num_options
#              small matmuls each preceded by a host sync.
DISPATCH = 'fused'

# Training recipe -- deliberately identical to Cole's ChunkMoE run so the
# two models can be compared without the recipe confounding the result.
DATASET      = 'tatsu-lab/alpaca'
MAX_LENGTH   = 512
BATCH_SIZE   = 4
GRAD_ACCUM   = 8
EPOCHS       = 1
LR           = 5e-5
SEED         = 42

# Where the trained parameters end up (in your Drive)
SAVE_DIR   = '/content/drive/MyDrive/gpt2-aag-alpaca'
OUTPUT_DIR = '/content/aag_checkpoints'   # local scratch during training

torch.manual_seed(SEED)

## Mount Google Drive

Mount **before** anything writes to a Drive path. Writing to
`/content/drive/...` while unmounted creates ordinary local folders and then
blocks the real mount.

In [3]:
import shutil
from google.colab import drive

if os.path.exists('/content/drive') and not os.path.exists('/content/drive/MyDrive'):
    shutil.rmtree('/content/drive')
    print('cleared a stray /content/drive')

drive.mount('/content/drive')
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('saving to:', SAVE_DIR)

Mounted at /content/drive
saving to: /content/drive/MyDrive/gpt2-aag-alpaca


## Load base GPT-2

In [4]:
tokenizer = AutoTokenizer.from_pretrained('openai-community/gpt2')
tokenizer.pad_token = tokenizer.eos_token

model = GPT2LMHeadModel.from_pretrained('openai-community/gpt2')
model.config.pad_token_id = tokenizer.eos_token_id
model.config.use_cache = False

baseline_params = sum(p.numel() for p in model.parameters())
print(f'base GPT-2 parameters: {baseline_params:,}')
print(model.transformer.h[0].mlp)

# kept aside so the dispatch benchmark can build a probe layer after the
# real blocks have already been replaced
base_mlp_for_probe = copy.deepcopy(model.transformer.h[0].mlp)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

base GPT-2 parameters: 124,439,808
GPT2MLP(
  (c_fc): Conv1D(nf=3072, nx=768)
  (c_proj): Conv1D(nf=768, nx=3072)
  (act): NewGELUActivation()
  (dropout): Dropout(p=0.1, inplace=False)
)


## Architecture

### `ChunkLinearMoE`

One Linear split into `num_chunks` output bands, each carrying `num_options`
candidate sub-matrices.

Three details matter, and each fixes a specific failure:

1. **Pretrained-slice initialisation.** Option `o` of band `c` starts from the
   matching row-slice of GPT-2's own MLP rather than from `randn`, so the
   layer inherits what GPT-2 already knows about language. A small amount of
   noise separates the options, otherwise they are identical and the router
   has no gradient signal to tell them apart.

2. **Fused dispatch.** Indexing the weight bank with a per-token index
   (`chunk_weights[c, idx]`) materialises an `[N, chunk_dim, in_features]`
   tensor -- about 17 GiB at batch 4 x seq 512. Masking per option instead
   fixes the memory but issues `num_chunks * num_options` tiny matmuls per
   layer, each preceded by a `mask.any()` that stalls the GPU on Python; that
   version ran at ~0.4% of an L4's throughput. Computing every option in one
   matmul and gathering the winner does `num_options` times the arithmetic in
   a single kernel, and is far faster because the bottleneck was dispatch,
   not compute. Both the extra arithmetic and the extra activation memory are
   `out_features * num_options`, **independent of `NUM_CHUNKS`** -- so 16
   chunks costs exactly what 1 chunk does.

3. **Gate normalisation.** Scaling a band by its router probability (~1/4 at
   initialisation) would shrink the output to a quarter of the pretrained
   MLP's, throwing away the head start point 1 just bought. Dividing by the
   detached probability leaves the forward pass numerically unchanged while
   still routing a gradient back to the router.

In [5]:
class ChunkLinearMoE(nn.Module):
    """A Linear whose output rows are split into independently routed bands."""

    def __init__(self, in_features, out_features, num_chunks, num_options,
                 pretrained_weight=None, pretrained_bias=None,
                 init_noise=0.02, preserve_init=True, dispatch=None):
        super().__init__()
        assert out_features % num_chunks == 0, (
            f'out_features ({out_features}) must divide by num_chunks ({num_chunks})')

        self.in_features = in_features
        self.out_features = out_features
        self.num_chunks = num_chunks
        self.num_options = num_options
        self.chunk_dim = out_features // num_chunks
        self.preserve_init = preserve_init
        self.dispatch = dispatch or DISPATCH

        self.router = nn.Linear(in_features, num_chunks * num_options)
        self.chunk_weights = nn.Parameter(
            torch.empty(num_chunks, num_options, self.chunk_dim, in_features))
        self.chunk_biases = nn.Parameter(
            torch.zeros(num_chunks, num_options, self.chunk_dim))

        self._init_weights(pretrained_weight, pretrained_bias, init_noise)

        self.saved_router_probs = None
        self.routing_log = []
        self.track_routing = False

    def _init_weights(self, weight, bias, noise):
        if weight is None:
            nn.init.normal_(self.chunk_weights, std=0.02)
            return

        # GPT-2 uses Conv1D, whose weight is stored [in_features, out_features].
        w = weight.t().contiguous()
        with torch.no_grad():
            scale = w.std() * noise
            for c in range(self.num_chunks):
                lo, hi = c * self.chunk_dim, (c + 1) * self.chunk_dim
                for o in range(self.num_options):
                    self.chunk_weights[c, o].copy_(w[lo:hi])
                    self.chunk_weights[c, o].add_(
                        torch.randn_like(self.chunk_weights[c, o]) * scale)
                    if bias is not None:
                        self.chunk_biases[c, o].copy_(bias[lo:hi])

    def forward(self, x):
        n_tokens = x.shape[0]

        router_logits = self.router(x).view(n_tokens, self.num_chunks, self.num_options)
        router_probs = F.softmax(router_logits, dim=-1)
        if self.training:
            self.saved_router_probs = router_probs

        top_weights, top_indices = router_probs.max(dim=-1)
        if self.track_routing:
            self.routing_log.append(top_indices.detach().cpu())

        if self.dispatch == 'fused':
            out = self._fused(x, top_indices, n_tokens)
        else:
            out = self._masked(x, top_indices, n_tokens)

        gate = top_weights.unsqueeze(-1)
        if self.preserve_init:
            gate = gate / gate.detach().clamp_min(1e-9)

        return (out * gate).reshape(n_tokens, self.out_features)

    def _fused(self, x, top_indices, n_tokens):
        """All options in one matmul, then gather the winner per band."""
        all_options = F.linear(
            x,
            self.chunk_weights.reshape(-1, self.in_features),
            self.chunk_biases.reshape(-1),
        ).view(n_tokens, self.num_chunks, self.num_options, self.chunk_dim)

        selection = top_indices[:, :, None, None].expand(
            n_tokens, self.num_chunks, 1, self.chunk_dim)
        return all_options.gather(2, selection).squeeze(2)

    def _masked(self, x, top_indices, n_tokens):
        """Only the needed arithmetic, but many small kernels and host syncs."""
        chunks = []
        for c in range(self.num_chunks):
            idx_c = top_indices[:, c]
            out_c = None
            for o in range(self.num_options):
                mask = idx_c == o
                if not mask.any():
                    continue
                option_out = F.linear(x[mask], self.chunk_weights[c, o],
                                      self.chunk_biases[c, o])
                if out_c is None:
                    # From the result, not from x: under autocast F.linear
                    # returns half while x is float, and an index-put needs
                    # both sides to share a dtype.
                    out_c = torch.zeros(n_tokens, self.chunk_dim,
                                        dtype=option_out.dtype,
                                        device=option_out.device)
                out_c[mask] = option_out
            if out_c is None:
                out_c = x.new_zeros(n_tokens, self.chunk_dim)
            chunks.append(out_c)

        return torch.stack(chunks, dim=1)

### `AAGLayer` -- the drop-in replacement for a GPT-2 block MLP

In [6]:
class AAGLayer(nn.Module):
    """Two chunked Linears with GPT-2's activation between them."""

    def __init__(self, original_mlp, num_chunks=8, num_options=4,
                 aux_loss_coef=0.05, preserve_init=True):
        super().__init__()
        self.num_chunks = num_chunks
        self.num_options = num_options
        self.aux_loss_coef = aux_loss_coef

        w_fc, b_fc = original_mlp.c_fc.weight, original_mlp.c_fc.bias
        w_proj, b_proj = original_mlp.c_proj.weight, original_mlp.c_proj.bias
        hidden_dim, intermediate_dim = w_fc.shape[0], w_fc.shape[1]

        self.c_fc = ChunkLinearMoE(hidden_dim, intermediate_dim, num_chunks,
                                   num_options, w_fc, b_fc,
                                   preserve_init=preserve_init)
        self.act = ACT2FN['gelu_new']
        self.c_proj = ChunkLinearMoE(intermediate_dim, hidden_dim, num_chunks,
                                     num_options, w_proj, b_proj,
                                     preserve_init=preserve_init)

    @property
    def track_routing(self):
        return self.c_fc.track_routing

    @track_routing.setter
    def track_routing(self, value):
        # Only the up-projection is logged, which keeps the log one row per
        # token so it lines up with the padding mask.
        self.c_fc.track_routing = value

    @property
    def routing_log(self):
        return self.c_fc.routing_log

    @routing_log.setter
    def routing_log(self, value):
        self.c_fc.routing_log = value
        self.c_proj.routing_log = []

    def forward(self, hidden_states, *args, **kwargs):
        shape = hidden_states.shape
        flat = hidden_states.view(-1, shape[-1])
        out = self.c_proj(self.act(self.c_fc(flat)))
        return out.view(shape)

### Which dispatch is faster on *this* GPU?

The two paths compute the same thing. `fused` does `NUM_OPTIONS` times the
arithmetic in one kernel; `masked` does the minimum arithmetic across
`NUM_CHUNKS * NUM_OPTIONS` small kernels, each preceded by a host
synchronisation.

Which wins depends on whether the layer is limited by arithmetic or by kernel
dispatch, and that is a property of the hardware -- on CPU `masked` wins, on a
dispatch-bound GPU `fused` should win by a lot. Two minutes here can save
hours, so measure rather than assume.

In [7]:
import copy as _copy

probe = AAGLayer(_copy.deepcopy(base_mlp_for_probe), num_chunks=NUM_CHUNKS,
                 num_options=NUM_OPTIONS).to(device)
probe_x = torch.randn(BATCH_SIZE * MAX_LENGTH, probe.c_fc.in_features,
                      device=device)


def time_dispatch(mode, reps=8):
    probe.c_fc.dispatch = probe.c_proj.dispatch = mode
    use_amp = torch.cuda.is_available()
    for _ in range(3):
        with torch.autocast('cuda', enabled=use_amp):
            probe(probe_x.unsqueeze(0)).sum().backward()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(reps):
        with torch.autocast('cuda', enabled=use_amp):
            probe(probe_x.unsqueeze(0)).sum().backward()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    return (time.perf_counter() - start) / reps * 1000


t_fused = time_dispatch('fused')
t_masked = time_dispatch('masked')

print(f'fused  : {t_fused:8.1f} ms per layer fwd+bwd')
print(f'masked : {t_masked:8.1f} ms per layer fwd+bwd')
print()
winner = 'fused' if t_fused < t_masked else 'masked'
print(f'faster here: {winner}  ({max(t_fused, t_masked) / min(t_fused, t_masked):.2f}x)')
print(f'estimated full run: '
      f'{min(t_fused, t_masked) * 12 * 3 * 1544 * GRAD_ACCUM / 1000 / 3600:.1f} h')
print()
print(f'set DISPATCH = {winner!r} in Config, then re-run from Config down.')

del probe, probe_x
if torch.cuda.is_available():
    torch.cuda.empty_cache()

fused  :     10.1 ms per layer fwd+bwd
masked :    100.5 ms per layer fwd+bwd

faster here: fused  (9.91x)
estimated full run: 1.3 h

set DISPATCH = 'fused' in Config, then re-run from Config down.


## Replace every block's MLP

In [8]:
for block in model.transformer.h:
    block.mlp = AAGLayer(block.mlp, num_chunks=NUM_CHUNKS,
                         num_options=NUM_OPTIONS, aux_loss_coef=AUX_COEF)

model.to(device)
print(model.transformer.h[0].mlp)

total = sum(p.numel() for p in model.parameters())
print()
print(f'parameters after AAG : {total:,}   (base GPT-2 was {baseline_params:,})')

AAGLayer(
  (c_fc): ChunkLinearMoE(
    (router): Linear(in_features=768, out_features=32, bias=True)
  )
  (act): NewGELUActivation()
  (c_proj): ChunkLinearMoE(
    (router): Linear(in_features=3072, out_features=32, bias=True)
  )
)

parameters after AAG : 295,922,688   (base GPT-2 was 124,439,808)


### What the swap bought

Stored parameters go up, but the arithmetic executed per token does not: one
option is picked per band and the bands partition the output dimension, so
exactly one full matrix runs.

In [9]:
mlp = model.transformer.h[0].mlp

stored = (mlp.c_fc.chunk_weights.numel() + mlp.c_fc.chunk_biases.numel()
          + mlp.c_proj.chunk_weights.numel() + mlp.c_proj.chunk_biases.numel())
active = (mlp.c_fc.out_features * mlp.c_fc.in_features + mlp.c_fc.out_features
          + mlp.c_proj.out_features * mlp.c_proj.in_features + mlp.c_proj.out_features)
per_projection = float(NUM_OPTIONS) ** NUM_CHUNKS

print(f'per block, stored MLP params : {stored:,}')
print(f'per block, active per token  : {active:,}   <- same as vanilla GPT-2')
print(f'virtual experts per block    : {per_projection ** 2:.3e}')
print(f'virtual experts, whole model : 1e{math.log10(per_projection ** 2) * 12:.1f}')

per block, stored MLP params : 18,889,728
per block, active per token  : 4,722,432   <- same as vanilla GPT-2
virtual experts per block    : 4.295e+09
virtual experts, whole model : 1e115.6


## Dataset

Standard Alpaca instruction formatting. Prompt tokens are masked out of the
labels with `-100`, so the loss is measured on the response only.

In [10]:
dataset = load_dataset(DATASET)
dataset = dataset['train'].train_test_split(test_size=0.05, seed=SEED)
print(dataset)

README.md:   0%|          | 0.00/7.47k [00:00<?, ?B/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…): reconstructing file:   0%|          |  0.00B / 24.2MB            

data/train-00000-of-00001-a09b74b3ef9c3b(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 49401
    })
    test: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 2601
    })
})


In [11]:
PROMPT_WITH_INPUT = (
    'Below is an instruction that describes a task, paired with an input that '
    'provides further context. Write a response that appropriately completes '
    'the request.',
)


def format_prompt(instruction, input_text=''):
    header = ('Below is an instruction that describes a task, paired with an '
              'input that provides further context. Write a response that '
              'appropriately completes the request.')
    header_no_input = ('Below is an instruction that describes a task. Write a '
                       'response that appropriately completes the request.')
    if input_text:
        return (header + chr(10) + chr(10)
                + '### Instruction:' + chr(10) + instruction + chr(10) + chr(10)
                + '### Input:' + chr(10) + input_text + chr(10) + chr(10)
                + '### Response:' + chr(10))
    return (header_no_input + chr(10) + chr(10)
            + '### Instruction:' + chr(10) + instruction + chr(10) + chr(10)
            + '### Response:' + chr(10))


def preprocess(examples):
    input_ids, attention_mask, labels = [], [], []

    for instruction, input_text, output in zip(examples['instruction'],
                                               examples['input'],
                                               examples['output']):
        prompt = format_prompt(instruction, input_text)
        full = prompt + output + tokenizer.eos_token

        prompt_ids = tokenizer(prompt, truncation=True, max_length=MAX_LENGTH)['input_ids']
        full_ids = tokenizer(full, truncation=True, max_length=MAX_LENGTH)['input_ids']

        label = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
        pad = MAX_LENGTH - len(full_ids)

        input_ids.append(full_ids + [tokenizer.pad_token_id] * pad)
        attention_mask.append([1] * len(full_ids) + [0] * pad)
        labels.append(label + [-100] * pad)

    return {'input_ids': input_ids, 'attention_mask': attention_mask,
            'labels': labels}


tokenized = dataset.map(preprocess, batched=True,
                        remove_columns=dataset['train'].column_names)

# Drop examples whose prompt filled MAX_LENGTH, leaving no supervised tokens.
tokenized = tokenized.filter(lambda ex: any(t != -100 for t in ex['labels']))
print(tokenized)

Map:   0%|          | 0/49401 [00:00<?, ? examples/s]

Map:   0%|          | 0/2601 [00:00<?, ? examples/s]

Filter:   0%|          | 0/49401 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2601 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 49400
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2601
    })
})


### Collate

Batching is done by an explicit collate rather than `set_format('torch')`.
That format routes through the datasets Torch formatter, which imports
`torchvision.io.VideoReader` -- an import that fails outright on the
torchvision build Colab currently ships. Every example is already padded to
`MAX_LENGTH`, so a plain stack is all that is required.

In [12]:
def collate(features):
    keys = ('input_ids', 'attention_mask', 'labels')
    return {k: torch.tensor([f[k] for f in features], dtype=torch.long)
            for k in keys}

## Training

### Load-balancing auxiliary loss

Without it the routers collapse onto one option per band and the combinatorial
capacity is nominal only -- the same expert-collapse failure the plain MoE hit.
Balance is enforced per band, across that band's options, averaged over bands.

In [13]:
def balance_term(probs, num_options):
    hard = probs.argmax(dim=-1)
    fraction = torch.bincount(hard, minlength=num_options).float() / hard.numel()
    mean_prob = probs.mean(dim=0)
    return num_options * torch.sum(fraction.to(probs.device) * mean_prob)


def auxiliary_loss(model, attention_mask=None):
    flat_mask = attention_mask.reshape(-1).bool() if attention_mask is not None else None
    total = 0.0

    for block in model.transformer.h:
        mlp = block.mlp
        for linear in (mlp.c_fc, mlp.c_proj):
            probs = linear.saved_router_probs
            if probs is None:
                continue
            if flat_mask is not None and flat_mask.shape[0] == probs.shape[0]:
                probs = probs[flat_mask]
            if not probs.numel():
                continue
            per_chunk = torch.stack([
                balance_term(probs[:, c, :], linear.num_options)
                for c in range(linear.num_chunks)])
            total = total + mlp.aux_loss_coef * per_chunk.mean()

    return total

### Trainer

The language-modelling term is delegated to `Trainer.compute_loss` rather than
recomputed. Trainer normalises across a gradient-accumulation window using
`num_items_in_batch`, and skips its own division by `gradient_accumulation_steps`
when it sees a loss computed that way. Calling `model(**inputs)` directly and
dropping that argument returns a per-batch mean that never gets divided, making
gradients -- and the effective learning rate -- `GRAD_ACCUM` times too large.

In [14]:
class AAGTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False,
                     num_items_in_batch=None, **kwargs):
        loss, outputs = super().compute_loss(
            model, inputs, return_outputs=True,
            num_items_in_batch=num_items_in_batch, **kwargs)

        if model.training:
            aux = auxiliary_loss(model, inputs.get('attention_mask'))
            if torch.is_tensor(aux):
                # The LM term is normalised over the whole accumulation window;
                # the aux term is per micro-batch, so divide by the same factor.
                loss = loss + aux / max(GRAD_ACCUM, 1)

        return (loss, outputs) if return_outputs else loss

In [15]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_steps=30,
    logging_steps=25,
    eval_strategy='no',
    save_strategy='steps',
    save_steps=250,
    save_total_limit=1,
    fp16=torch.cuda.is_available(),
    report_to='none',
    seed=SEED,
)

trainer = AAGTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    data_collator=collate,
)

### Run it

Roughly 2-3 hours on an L4 for one epoch of Alpaca.

**Sanity check the first logged loss.** GPT-2 is pretrained, so it should be
around **2.5-3.5**, with `grad_norm` around 1-5. A loss above 10 means
something is wrong -- `ln(50257) = 10.8` is what an untrained model scores, so
anything above that is worse than guessing uniformly.

In [16]:
model.train()
result = trainer.train()
print(result.metrics)

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
25,3.857742
50,3.544755
75,3.458024
100,3.458031
125,3.444945
150,3.412790
175,3.398150
200,3.409984
225,3.414401
250,3.417211


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': 3620.0146, 'train_samples_per_second': 13.646, 'train_steps_per_second': 0.427, 'total_flos': 3.8931519504384e+16, 'train_loss': 3.3186570523316377, 'epoch': 1.0}


## Save the trained parameters to Drive

In [17]:
model.save_pretrained(SAVE_DIR, safe_serialization=True)
tokenizer.save_pretrained(SAVE_DIR)

print('saved to', SAVE_DIR)
for name in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(os.path.join(SAVE_DIR, name)) / 1e6
    print(f'   {name:28s} {size:9.1f} MB')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved to /content/drive/MyDrive/gpt2-aag-alpaca
   config.json                        0.0 MB
   generation_config.json             0.0 MB
   model.safetensors               1183.7 MB
   tokenizer.json                     3.6 MB
   tokenizer_config.json              0.0 MB


> To reload this model later, re-run the Import, Config, Architecture and
> *Replace every block's MLP* cells, then:
> ```python
> from safetensors.torch import load_file
> model.load_state_dict(load_file(SAVE_DIR + '/model.safetensors'), strict=False)
> ```

## Evaluation

### Perplexity on the held-out split

Token-weighted over supervised tokens only, so it measures response quality
rather than the model's ability to echo the prompt template.

In [18]:
@torch.no_grad()
def evaluate_perplexity(model, dataset, batch_size=8):
    model.eval()
    loader = DataLoader(dataset, batch_size=batch_size, collate_fn=collate)

    total_nll, total_tokens = 0.0, 0
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        logits = model(input_ids=batch['input_ids'],
                       attention_mask=batch['attention_mask']).logits

        shift_logits = logits[:, :-1, :].reshape(-1, logits.size(-1))
        shift_labels = batch['labels'][:, 1:].reshape(-1)

        total_nll += F.cross_entropy(shift_logits.float(), shift_labels,
                                     ignore_index=-100, reduction='sum').item()
        total_tokens += int((shift_labels != -100).sum())

    nll = total_nll / max(total_tokens, 1)
    return {'eval_tokens': total_tokens, 'loss': nll, 'perplexity': math.exp(nll)}


metrics = evaluate_perplexity(model, tokenized['test'])
print(f"eval tokens : {metrics['eval_tokens']:,}")
print(f"loss        : {metrics['loss']:.4f}")
print(f"perplexity  : {metrics['perplexity']:.3f}")

eval tokens : 146,063
loss        : 1.9816
perplexity  : 7.254


### Routing health

`entropy / max` is 1.0 when every option in a band gets equal use and 0.0 when
the router has collapsed onto one. Collapse would mean the combinatorial
capacity exists on paper only, so this is the number that decides whether the
whole approach held up.

In [19]:
@torch.no_grad()
def routing_health(model, dataset, batch_size=8, max_batches=20):
    model.eval()
    loader = DataLoader(dataset, batch_size=batch_size, collate_fn=collate)
    counts = {i: torch.zeros(NUM_OPTIONS) for i in range(len(model.transformer.h))}

    for n, batch in enumerate(loader):
        if n >= max_batches:
            break
        for block in model.transformer.h:
            block.mlp.routing_log = []
            block.mlp.track_routing = True

        model(input_ids=batch['input_ids'].to(device),
              attention_mask=batch['attention_mask'].to(device))

        keep = batch['attention_mask'].reshape(-1).bool()
        for i, block in enumerate(model.transformer.h):
            block.mlp.track_routing = False
            logs = block.mlp.routing_log
            if not logs:
                continue
            sel = torch.cat(logs, dim=0)
            if sel.shape[0] == keep.shape[0]:
                sel = sel[keep]
            counts[i] += torch.bincount(sel.reshape(-1),
                                        minlength=NUM_OPTIONS).float()

    max_entropy = math.log2(NUM_OPTIONS)
    print('Layer | ' + ' | '.join(f'opt {o}' for o in range(NUM_OPTIONS))
          + ' | entropy/max')
    print('-' * 62)

    ratios = []
    for i in range(len(model.transformer.h)):
        share = counts[i] / counts[i].sum().clamp_min(1)
        nonzero = share[share > 0]
        entropy = float(-(nonzero * nonzero.log2()).sum())
        ratios.append(entropy / max_entropy)
        pct = ' | '.join(f'{100 * s:5.1f}%' for s in share)
        print(f'L{i:02d}   | {pct} | {entropy / max_entropy:.3f}')

    print('-' * 62)
    print(f'mean entropy ratio: {sum(ratios) / len(ratios):.3f}   (1.0 = balanced, 0.0 = collapsed)')


routing_health(model, tokenized['test'])

Layer | opt 0 | opt 1 | opt 2 | opt 3 | entropy/max
--------------------------------------------------------------
L00   |  25.1% |  23.9% |  26.6% |  24.4% | 0.999
L01   |  24.4% |  23.8% |  25.7% |  26.2% | 0.999
L02   |  26.5% |  24.1% |  24.8% |  24.6% | 1.000
L03   |  23.9% |  25.9% |  25.7% |  24.6% | 1.000
L04   |  25.8% |  24.9% |  24.8% |  24.5% | 1.000
L05   |  25.6% |  24.8% |  25.6% |  24.0% | 1.000
L06   |  24.9% |  24.6% |  25.6% |  24.9% | 1.000
L07   |  25.1% |  24.9% |  24.5% |  25.4% | 1.000
L08   |  25.6% |  23.3% |  25.5% |  25.7% | 0.999
L09   |  24.7% |  24.6% |  25.2% |  25.6% | 1.000
L10   |  24.2% |  25.1% |  25.6% |  25.2% | 1.000
L11   |  25.4% |  25.2% |  24.3% |  25.1% | 1.000
--------------------------------------------------------------
mean entropy ratio: 1.000   (1.0 = balanced, 0.0 = collapsed)


### Sample generations

In [20]:
@torch.no_grad()
def generate(instruction, input_text='', max_new_tokens=60):
    model.eval()
    model.config.use_cache = True
    inputs = tokenizer(format_prompt(instruction, input_text),
                       return_tensors='pt').to(device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True,
                         top_k=40, top_p=0.9, temperature=0.6,
                         pad_token_id=tokenizer.eos_token_id)
    model.config.use_cache = False
    text = tokenizer.decode(out[0][inputs.input_ids.shape[1]:],
                            skip_special_tokens=True).strip()
    print('PROMPT  :', instruction)
    if input_text:
        print('INPUT   :', input_text)
    print('RESPONSE:', text)
    print()


generate('What is the capital of Canada?')
generate('List three healthy snacks.')
generate('Correct the grammar in the sentence.', 'He do not have no money.')
generate('Write a short poem about a quiet rainy afternoon.')

PROMPT  : What is the capital of Canada?
RESPONSE: The capital of Canada is Ottawa.

PROMPT  : List three healthy snacks.
RESPONSE: Three healthy snacks are yogurt, granola bars, and ice cream.

PROMPT  : Correct the grammar in the sentence.
INPUT   : He do not have no money.
RESPONSE: He does not have money.

PROMPT  : Write a short poem about a quiet rainy afternoon.
RESPONSE: The sun was shining,
Filling the sky with a warm glow.
The air was warm,
Soft and peaceful,
With a gentle breeze of sweet fragrance.

The trees twinkled,
The birds singing,
The birds singing in a gentle way.
The wind



### Summary for the report

In [21]:
print('AAG on GPT-2, Alpaca, 1 epoch')
print(f'  chunks x options        : {NUM_CHUNKS} x {NUM_OPTIONS}')
print(f'  stored MLP params/block : {stored:,}')
print(f'  active per token/block  : {active:,}')
print(f'  virtual experts (model) : 1e{math.log10(float(NUM_OPTIONS) ** NUM_CHUNKS * float(NUM_OPTIONS) ** NUM_CHUNKS) * 12:.1f}')
print(f"  eval perplexity         : {metrics['perplexity']:.3f}")
print(f'  weights                 : {SAVE_DIR}')

AAG on GPT-2, Alpaca, 1 epoch
  chunks x options        : 8 x 4
  stored MLP params/block : 18,889,728
  active per token/block  : 4,722,432
  virtual experts (model) : 1e115.6
  eval perplexity         : 7.254
  weights                 : /content/drive/MyDrive/gpt2-aag-alpaca
